Joins, Datetime operations, Window Functions and File handling in pyspark

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("joins").getOrCreate()

your 131072x1 screen size is bogus. expect trouble
25/07/25 10:34:49 WARN Utils: Your hostname, IQT-RajenderMudasthu resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/07/25 10:34:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/25 10:34:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/07/25 10:34:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [10]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [14]:
emp_df = spark.read.csv("/home/rajender/pyspark/pyspark/employee.csv", header=True, inferSchema=True)
emp_df.show(3)

+------+---------------+---------------+---------------------+-----------+------+--------+
|emp_id|           name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|
+------+---------------+---------------+---------------------+-----------+------+--------+
|     1|           NULL|           NULL|  2016-06-15 04:37:53|        7.0| Other|111107.9|
|     2|Jason Rodriguez|          171.0|  2020-02-13 02:31:07|        1.0|Female|80486.58|
|     3|Justin Peterson|          224.0|  2016-03-18 21:29:35|        7.0|  Male|    NULL|
+------+---------------+---------------+---------------------+-----------+------+--------+
only showing top 3 rows



In [15]:
dep_df = spark.read.csv("/home/rajender/pyspark/pyspark/departments.csv", header=True, inferSchema=True)
dep_df.show(2)

+-------+---------+
|dept_id|dept_name|
+-------+---------+
|      1|       HR|
|      2|  Finance|
+-------+---------+
only showing top 2 rows



In [16]:
emp_df.count()

300

In [17]:
emp_df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- superior_emp_id: double (nullable = true)
 |-- joined_date_timestamp: timestamp (nullable = true)
 |-- emp_dept_id: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)



In [18]:
dep_df.printSchema()

root
 |-- dept_id: integer (nullable = true)
 |-- dept_name: string (nullable = true)



In [19]:
# change the datatypes of the emp_id and dept_id 
emp_df = emp_df.withColumn("emp_dept_id",col("emp_dept_id").cast(IntegerType()))
emp_df = emp_df.withColumn("superior_emp_id",col("superior_emp_id").cast(IntegerType()))

In [20]:
emp_df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- superior_emp_id: integer (nullable = true)
 |-- joined_date_timestamp: timestamp (nullable = true)
 |-- emp_dept_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)



In [25]:
#INNER JOIN
inner_df = emp_df.join(dep_df, emp_df.emp_dept_id == dep_df.dept_id, "inner")
inner_df.show(3)

+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
|emp_id|           name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|dept_id|dept_name|
+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
|     1|           NULL|           NULL|  2016-06-15 04:37:53|          7| Other|111107.9|      7|  Support|
|     2|Jason Rodriguez|            171|  2020-02-13 02:31:07|          1|Female|80486.58|      1|       HR|
|     3|Justin Peterson|            224|  2016-03-18 21:29:35|          7|  Male|    NULL|      7|  Support|
+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
only showing top 3 rows



In [26]:
inner_df.count()

260

In [31]:
# to check the count of nulls in each column in emp_df
total_nulls = emp_df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in emp_df.columns])
total_nulls.show()

+------+----+---------------+---------------------+-----------+------+------+
|emp_id|name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|salary|
+------+----+---------------+---------------------+-----------+------+------+
|     0|  12|             40|                   15|         40|    66|    30|
+------+----+---------------+---------------------+-----------+------+------+



In [ ]:
# to check the count of nulls in each column in dep_df
total_nulls_dept = dep_df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in dep_df.columns])
total_nulls_dept.show()

+-------+---------+
|dept_id|dept_name|
+-------+---------+
|      0|        0|
+-------+---------+



In [32]:
#OUTER JOIN
outer_df = emp_df.join(dep_df, emp_df.emp_dept_id == dep_df.dept_id, "outer")
outer_df.show(3)

+------+------------+---------------+---------------------+-----------+------+--------+-------+---------+
|emp_id|        name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|dept_id|dept_name|
+------+------------+---------------+---------------------+-----------+------+--------+-------+---------+
|    25|  Luke Dixon|            132|  2022-09-18 19:49:58|       NULL| Other|50173.48|   NULL|     NULL|
|    26|Steven Flynn|              4|  2017-01-27 12:46:02|       NULL| Other|41008.47|   NULL|     NULL|
|    31|David Miller|            229|  2018-01-05 12:04:57|       NULL| Other|92142.14|   NULL|     NULL|
+------+------------+---------------+---------------------+-----------+------+--------+-------+---------+
only showing top 3 rows



In [33]:
outer_df.count()

300

In [35]:
#Left Join
left_df = emp_df.join(dep_df, emp_df.emp_dept_id == dep_df.dept_id, "left")
left_df.show(3)

+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
|emp_id|           name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|dept_id|dept_name|
+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
|     1|           NULL|           NULL|  2016-06-15 04:37:53|          7| Other|111107.9|      7|  Support|
|     2|Jason Rodriguez|            171|  2020-02-13 02:31:07|          1|Female|80486.58|      1|       HR|
|     3|Justin Peterson|            224|  2016-03-18 21:29:35|          7|  Male|    NULL|      7|  Support|
+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
only showing top 3 rows



In [36]:
left_df.count()

300

In [37]:
#left semi join
left_semi_df = emp_df.join(dep_df, emp_df.emp_dept_id == dep_df.dept_id, "leftsemi")
left_semi_df.show(3)

+------+---------------+---------------+---------------------+-----------+------+--------+
|emp_id|           name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|
+------+---------------+---------------+---------------------+-----------+------+--------+
|     1|           NULL|           NULL|  2016-06-15 04:37:53|          7| Other|111107.9|
|     2|Jason Rodriguez|            171|  2020-02-13 02:31:07|          1|Female|80486.58|
|     3|Justin Peterson|            224|  2016-03-18 21:29:35|          7|  Male|    NULL|
+------+---------------+---------------+---------------------+-----------+------+--------+
only showing top 3 rows



In [38]:
left_semi_df.count()

260

In [39]:
#left anti join
left_anti_df = emp_df.join(dep_df, emp_df.emp_dept_id == dep_df.dept_id, "leftanti")
left_anti_df.show(3)

+------+------------+---------------+---------------------+-----------+------+--------+
|emp_id|        name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|
+------+------------+---------------+---------------------+-----------+------+--------+
|    25|  Luke Dixon|            132|  2022-09-18 19:49:58|       NULL| Other|50173.48|
|    26|Steven Flynn|              4|  2017-01-27 12:46:02|       NULL| Other|41008.47|
|    31|David Miller|            229|  2018-01-05 12:04:57|       NULL| Other|92142.14|
+------+------------+---------------+---------------------+-----------+------+--------+
only showing top 3 rows



In [40]:
left_anti_df.count()

40

In [41]:
#self join
self_df = emp_df.alias("emp1").join(emp_df.alias("emp2"), col("emp1.superior_emp_id") == col("emp2.emp_id"), "inner" )
self_df.show()

+------+------------------+---------------+---------------------+-----------+------+---------+------+--------------------+---------------+---------------------+-----------+------+---------+
|emp_id|              name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|   salary|emp_id|                name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|   salary|
+------+------------------+---------------+---------------------+-----------+------+---------+------+--------------------+---------------+---------------------+-----------+------+---------+
|     2|   Jason Rodriguez|            171|  2020-02-13 02:31:07|          1|Female| 80486.58|   171|    Alexandra Tucker|            295|  2017-10-01 12:33:47|          5|  NULL|116232.57|
|     3|   Justin Peterson|            224|  2016-03-18 21:29:35|          7|  Male|     NULL|   224|       Joseph Garcia|            294|  2019-11-23 04:44:51|          5|  NULL|     NULL|
|     4|       George Cook|             23|  2016-

In [ ]:
#to select multiple columns in a dataframe
self_df.select(col("emp1.emp_id"), col("emp1.name"), col("emp1.superior_emp_id").alias("manager_id"), col("emp2.name").alias("manager_name")).show(5)

+------+---------------+----------+----------------+
|emp_id|           name|manager_id|    manager_name|
+------+---------------+----------+----------------+
|     2|Jason Rodriguez|       171|Alexandra Tucker|
|     3|Justin Peterson|       224|   Joseph Garcia|
|     4|    George Cook|        23|        Chad Lee|
|     5|   Taylor Ayala|       136|   Amber Jenkins|
|     6|     Ethan West|       270|   Darin Nichols|
+------+---------------+----------+----------------+
only showing top 5 rows



In [45]:
emp_df.filter(col("emp_id") == 171).show()

+------+----------------+---------------+---------------------+-----------+------+---------+
|emp_id|            name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|   salary|
+------+----------------+---------------+---------------------+-----------+------+---------+
|   171|Alexandra Tucker|            295|  2017-10-01 12:33:47|          5|  NULL|116232.57|
+------+----------------+---------------+---------------------+-----------+------+---------+

